In [21]:
import pandas as pd
import numpy as np
import os

os.listdir(".")

['brfss_env',
 'analysis-brfss.ipynb',
 'LLCP2024.XPT',
 '.DS_Store',
 'income_race_analysis.ipynb',
 'LLCP2021.XPT',
 'LLCP2020.XPT',
 'LLCP2022.XPT',
 'LLCP2023.XPT',
 'brfss_clean_2020_2024.csv.zip',
 'brfss_clean_2020_2024.csv',
 'brfss_preprocess.ipynb',
 'brfss_group_summary.csv']

In [22]:
files = {
    2020: "LLCP2020.XPT",
    2021: "LLCP2021.XPT",
    2022: "LLCP2022.XPT",
    2023: "LLCP2023.XPT",
    2024: "LLCP2024.XPT"
}

for year, f in files.items():
    df = pd.read_sas(f, format="xport", encoding="latin1")
    print(f"{year}: {df.shape}")

2020: (401958, 279)
2021: (438693, 303)
2022: (445132, 328)
2023: (433323, 350)
2024: (457670, 301)


In [23]:
for year, f in files.items():
    df_temp = pd.read_sas(f, format="xport", encoding="latin1")
    income_cols = [c for c in df_temp.columns if "INCOM" in c]
    race_cols = [c for c in df_temp.columns if "RACE" in c]
    print(f"\n{year}:")
    print(f"  Income cols: {income_cols}")
    print(f"  Race cols: {race_cols}")


2020:
  Income cols: ['INCOME2', '_INCOMG']
  Race cols: ['_IMPRACE', '_CRACE1', '_CPRACE', '_PRACE1', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']

2021:
  Income cols: ['INCOME3', '_INCOMG1']
  Race cols: ['_IMPRACE', '_CRACE1', '_CPRACE1', '_PRACE1', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']

2022:
  Income cols: ['INCOME3', '_INCOMG1']
  Race cols: ['_IMPRACE', '_CRACE2', '_CPRACE2', '_PRACE2', '_MRACE2', '_RACE1', '_RACEG22', '_RACEGR4', '_RACEPR1']

2023:
  Income cols: ['INCOME3', '_INCOMG1']
  Race cols: ['_IMPRACE', '_CRACE1', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']

2024:
  Income cols: ['INCOME3', '_INCOMG1']
  Race cols: ['_IMPRACE', '_CRACE1', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']


In [35]:
def preprocess_brfss(file, year):

    df = pd.read_sas(file, format="xport", encoding="latin1")

    # Fix income variable name (2020 used _INCOMG, 2021+ uses _INCOMG1)
    if year == 2020:
        df = df.rename(columns={"_INCOMG": "_INCOMG1"})
        df = df[df["_INCOMG1"] < 9]
        df["_INCOMG1"] = np.nan

    # Fix race variable
    # 2022 uses _RACEPR1 (7 categories, Hispanic=7, Other/Multiracial=6)
    # All other years use _RACEPRV (8 categories, Hispanic=8, Other=6, Multiracial=7)
    # We harmonize to 7 categories across all years:
    # 1=NH-White, 2=NH-Black, 3=AIAN, 4=Asian, 5=NHOPI, 6=Other/Multiracial, 7=Hispanic
    if year == 2022:
        df = df.rename(columns={"_RACEPR1": "_RACEPRV"})
        # 2022 is already in the right format, no recoding needed
    else:
        # Recode other years to match 2022 scheme
        # Multiracial (7) -> Other/Multiracial (6)
        # Hispanic (8) -> 7
        df["_RACEPRV"] = df["_RACEPRV"].replace({7: 6, 8: 7})

    cols_needed = [
        "_STATE",
        "_LLCPWT",
        "_BMI5",
        "_AGEG5YR",
        "_SEX",
        "_EDUCAG",
        "_INCOMG1",
        "_RACEPRV"
    ]

    df = df[cols_needed]

    # Remove invalid BMI first before any calculation
    df = df[(df["_BMI5"] < 9000) & (df["_BMI5"] > 1200)]


    # Convert BMI
    df["BMI"] = df["_BMI5"] / 100

    # Obesity indicator
    df["obese"] = (df["BMI"] >= 30).astype(int)


    # Remove missing/refused demographics
    # _INCOMG1: valid codes 1-7, code 9 = missing
    # _RACEPRV: valid codes 1-7, code 9 = missing
    df = df[
        (df["_AGEG5YR"] < 14) &
        (df["_SEX"] < 3) &
        (df["_EDUCAG"] < 5) &
        (df["_RACEPRV"] < 8) &
        ((df["_INCOMG1"] < 8) | (year == 2020))
    ]

    df["year"] = year

    return df

In [36]:
brfss_2020 = preprocess_brfss("LLCP2020.XPT", 2020)
brfss_2021 = preprocess_brfss("LLCP2021.XPT", 2021)
brfss_2022 = preprocess_brfss("LLCP2022.XPT", 2022)
brfss_2023 = preprocess_brfss("LLCP2023.XPT", 2023)
brfss_2024 = preprocess_brfss("LLCP2024.XPT", 2024)

print(brfss_2020.shape)
print(brfss_2021.shape)
print(brfss_2022.shape)
print(brfss_2023.shape)
print(brfss_2024.shape)

(300259, 11)
(320968, 11)
(326682, 11)
(326419, 11)
(348171, 11)


In [37]:
brfss_all = pd.concat([
    brfss_2020,
    brfss_2021,
    brfss_2022,
    brfss_2023,
    brfss_2024
], ignore_index=True)

print("Total shape:", brfss_all.shape)

# Key check - race codes by year should all be 1-7 only
print("\n_RACEPRV value counts by year:")
print(brfss_all.groupby("year")["_RACEPRV"].value_counts().unstack().fillna(0).astype(int))

# Income check
print("\n_INCOMG1 value counts:")
print(brfss_all["_INCOMG1"].value_counts().sort_index())

Total shape: (1622499, 11)

_RACEPRV value counts by year:
_RACEPRV     1.0    2.0   3.0   4.0   5.0    6.0    7.0
year                                                   
2020      229065  22338  5244  7398  1611   8831  25772
2021      245069  23704  5533  8132  1497   9868  27165
2022      247265  25654  5294  9366  1855   7473  29775
2023      245139  24580  5268  8842  1658  10411  30521
2024      260491  26495  5078  9078  1636  11061  34332

_INCOMG1 value counts:
_INCOMG1
1.0     76637
2.0    124764
3.0    154172
4.0    179813
5.0    409021
6.0    283629
7.0     94204
Name: count, dtype: int64


In [39]:
brfss_all.to_csv("brfss_clean_2020_2024.csv", index=False)
print("saved.")

saved.


In [40]:
race_map = {
    1: "NH-White",
    2: "NH-Black",
    3: "AIAN",
    4: "Asian",
    5: "NHOPI",
    6: "Other/Multiracial",
    7: "Hispanic"
}

age_map = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39",
    5: "40-44", 6: "45-49", 7: "50-54", 8: "55-59",
    9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"
}

sex_map = {1: "Male", 2: "Female"}

education_map = {
    1: "Did not graduate high school",
    2: "Graduated high school",
    3: "Attended college or technical school",
    4: "Graduated college or technical school"
}

income_map = {
    1: "<15k", 2: "15k-25k", 3: "25k-35k",
    4: "35k-50k", 5: "50k-100k", 6: "100k-200k", 7: "200k+"
}

brfss_all["age_group"]    = brfss_all["_AGEG5YR"].map(age_map)
brfss_all["sex"]          = brfss_all["_SEX"].map(sex_map)
brfss_all["education"]    = brfss_all["_EDUCAG"].map(education_map)
brfss_all["income_group"] = brfss_all["_INCOMG1"].map(income_map)
brfss_all["race_group"]   = brfss_all["_RACEPRV"].map(race_map)

print("Race groups:", sorted(brfss_all["race_group"].dropna().unique()))
print("Income groups:", sorted(brfss_all["income_group"].dropna().unique()))
print("Age groups:", sorted(brfss_all["age_group"].dropna().unique()))

Race groups: ['AIAN', 'Asian', 'Hispanic', 'NH-Black', 'NH-White', 'NHOPI', 'Other/Multiracial']
Income groups: ['100k-200k', '15k-25k', '200k+', '25k-35k', '35k-50k', '50k-100k', '<15k']
Age groups: ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']


In [41]:
group_cols = ["age_group", "sex", "education", "income_group", "race_group"]

def weighted_obesity_rate(group):
    return np.average(group["obese"], weights=group["_LLCPWT"])

brfss_group_summary = (
    brfss_all.groupby(group_cols)
      .apply(weighted_obesity_rate)
      .reset_index()
)
brfss_group_summary.columns = group_cols + ["obesity_rate"]

# Add unweighted n for sample size reference
n_counts = brfss_all.groupby(group_cols).size().reset_index(name="n")
brfss_group_summary = brfss_group_summary.merge(n_counts, on=group_cols)
brfss_group_summary = brfss_group_summary.sort_values("n", ascending=False)

print(brfss_group_summary.shape)
brfss_group_summary.to_csv("brfss_group_summary.csv", index=False)
print("saved.")

(4953, 7)
saved.


In [42]:
df_check = pd.read_csv("brfss_clean_2020_2024.csv")
print(df_check.columns.tolist())
print(df_check["BMI"].describe())
print(df_check["BMI"].head(10))

['_STATE', '_LLCPWT', '_BMI5', '_AGEG5YR', '_SEX', '_EDUCAG', '_INCOMG1', '_RACEPRV', 'BMI', 'obese', 'year']
count    1.622499e+06
mean     2.866438e+01
std      6.519412e+00
min      1.202000e+01
25%      2.428000e+01
50%      2.746000e+01
75%      3.189000e+01
max      8.997000e+01
Name: BMI, dtype: float64
0    16.60
1    26.58
2    24.21
3    26.57
4    23.71
5    21.63
6    31.64
7    26.45
8    40.69
9    28.71
Name: BMI, dtype: float64


In [43]:
df_check = pd.read_csv("brfss_clean_2020_2024.csv")

print("Shape:", df_check.shape)

print("\n_INCOMG1 by year (NaN count shows 2020 is correctly nulled):")
print(df_check.groupby("year")["_INCOMG1"].apply(lambda x: x.isna().sum()).rename("NaN count"))
print(df_check.groupby("year")["_INCOMG1"].apply(lambda x: x.notna().sum()).rename("Valid count"))

print("\n_INCOMG1 value counts (should have no values for 2020):")
print(df_check.groupby("year")["_INCOMG1"].value_counts().unstack().fillna(0).astype(int))

print("\nRace codes by year (should all be 1-7):")
print(df_check.groupby("year")["_RACEPRV"].value_counts().unstack().fillna(0).astype(int))

print("\nBMI range:")
print(df_check["BMI"].describe())

Shape: (1622499, 11)

_INCOMG1 by year (NaN count shows 2020 is correctly nulled):
year
2020    300259
2021         0
2022         0
2023         0
2024         0
Name: NaN count, dtype: int64
year
2020         0
2021    320968
2022    326682
2023    326419
2024    348171
Name: Valid count, dtype: int64

_INCOMG1 value counts (should have no values for 2020):
_INCOMG1    1.0    2.0    3.0    4.0     5.0    6.0    7.0
year                                                      
2021      20608  33153  40533  45105  100279  63317  17973
2022      19573  32071  39141  43633  101164  68811  22289
2023      17508  28634  35789  44573  101300  73041  25574
2024      18948  30906  38709  46502  106278  78460  28368

Race codes by year (should all be 1-7):
_RACEPRV     1.0    2.0   3.0   4.0   5.0    6.0    7.0
year                                                   
2020      229065  22338  5244  7398  1611   8831  25772
2021      245069  23704  5533  8132  1497   9868  27165
2022      247265  2